# Kaggle launcher (THIN)

Pulls a pinned commit of [Nafish32/pulmonary](https://github.com/Nafish32/pulmonary) and runs `src.pipeline.run_all`. No business logic here.

**Before running:**
- **Add data** (right panel): attach the **RSNA** dataset (`stage_2_train_labels.csv` + `stage_2_train_images/`). Attach **VinDr/VinBigData** too for the external-validation stage (optional — it skips cleanly if absent).
- Settings: **GPU on** (T4×2 recommended — `thesis.yaml` uses both via DDP), **Internet on**, **Persistence: Variables and Files** so a killed run can resume.

**Run order:** Cell 1 (clone) → Cell 2 (pick config) → Cell 3 (`run_all`) → Cell 4 (report). A full `thesis.yaml` run is ~6–7 hr on T4×2 (~12 hr single-GPU) and produces every trust-package number. If the session is killed mid-run, just re-run Cells 1→3 in the **same session** — it resumes from the last checkpoint, no retrain. The **Debug** and **Eval-only** sections at the bottom are optional side-paths.

In [ ]:
# 1) Pull pinned commit + install (idempotent: safe to re-run in the same session)
import os, subprocess

REPO_URL = "https://github.com/Nafish32/pulmonary.git"
COMMIT   = "cbbb027"   # thesis-numbers run pinned to this SHA. Use "main" to track latest.
REPO_DIR = "/kaggle/working/repo"

# CRITICAL: step out of REPO_DIR before deleting it. A prior run left the process
# CWD inside REPO_DIR; rm -rf on the CWD gives 'getcwd: cannot access parent directories'
# and every shell command after that silently no-ops.
os.chdir("/kaggle/working")

def sh(cmd):
    print(">", cmd)
    subprocess.run(cmd, shell=True, check=True, cwd="/kaggle/working")

sh(f"rm -rf {REPO_DIR}")
sh(f"git clone --quiet {REPO_URL} {REPO_DIR}")
sh(f"git -C {REPO_DIR} checkout --quiet {COMMIT}")
sh(f"pip install -q -r {REPO_DIR}/requirements.txt")
sh(f"pip install -q -e {REPO_DIR}")

# print the exact SHA that ran, so any number is traceable even when COMMIT='main'
sha = subprocess.check_output(f"git -C {REPO_DIR} rev-parse HEAD", shell=True).decode().strip()
print("[GIT] running commit:", sha)

In [ ]:
# 2) Load + validate config (fails fast, prints fast_mode loudly)
import os, logging, sys
os.chdir("/kaggle/working/repo")
logging.basicConfig(level=logging.INFO, stream=sys.stdout, force=True)  # make src logs visible + live

# Drop stale src.* modules: Cell 1 re-clones the repo, but a re-run without a
# kernel restart keeps old code cached in sys.modules (traceback shows new file
# lines while the OLD code object runs). Pop them so imports below load fresh.
for m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    sys.modules.pop(m)

from src.config.loader import load_config

# THESIS-NUMBERS run (full RSNA, 50 ep, both GPUs, all trust stages).
# Smoke the chain first with fast.yaml if you changed anything risky.
cfg = load_config("configs/thesis.yaml")
# cfg = load_config("configs/fast.yaml")
print("fast_mode =", cfg.fast_mode, "| epochs =", cfg.epochs,
      "| max_patients =", cfg.max_patients, "| device =", repr(cfg.device))

In [ ]:
# 3) Run everything end to end.
# First stages (discover rglob, DICOM caching) are IO-bound and can run minutes
# with near-zero CPU/GPU and little output -- that is normal, not a hang.
from src.pipeline import run_all
results_md = run_all(cfg)

In [ ]:
# 4) Report
print("results:", results_md)
from IPython.display import Markdown, display
with open(results_md) as f:
    display(Markdown(f.read()))

## Debug: run stages one at a time

If Cell 3 sits silent and you want to see where, interrupt it and run this. Each line returns and prints, so a slow stage is obvious. Also confirms both GPUs are visible before the long run.

In [ ]:
from src.data.discover import discover_datasets
ds = discover_datasets(cfg.input_root); print("DISCOVER OK", ds)  # slow here == rglob over input mount

import pandas as pd
df = pd.read_csv(ds["rsna_csv"]); print("rows", len(df), "patients", df.patientId.nunique())

import torch
print("cuda:", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count())  # 2 -> DDP will use both
# 0 GPUs -> Accelerator not actually on; 1 -> set thesis.yaml device to "0"; 2 -> "0,1" uses both

## Eval-only (recovery / re-score)

Re-score an existing `best.pt` **without** the ~12 hr train. Use it to recover a run whose training finished but eval was wrong, or to score different weights. Runs the same data prep + all trust stages (minus ensemble spread, which needs the members `run_all` trains). ~7 min. Requires Cell 1 + Cell 2 first (repo pulled, `cfg` loaded).

In [ ]:
from src.pipeline import eval_from_weights
from IPython.display import Markdown, display

# From a same-session finished train:
WEIGHTS = "/kaggle/working/outputs/runs/train_thesis/weights/best.pt"
# Or an uploaded weights dataset:
# WEIGHTS = "/kaggle/input/<your-best-pt-dataset>/best.pt"

results_md = eval_from_weights(cfg, WEIGHTS)
print("results:", results_md)
display(Markdown(open(results_md).read()))